# 3.2 — bigger pretrained backbones, fine-tuned

Fine-tuning worked. `v30-finetune_encoder_1e-5` — ResNet18, encoder at 1e-5 while the head
runs at 7e-4 — scored **0.8938** [0.8799, 0.9049] with `Scratch` at **0.813**, the best
`Scratch` in the project, against **0.7351** for the same backbone frozen. **+0.159 from
unfreezing.** The pretrained weights are worth something; they just have to be allowed to
move.

It also hit its epoch cap — best at 31 of 40 with patience 10, so early stopping never
fired and 0.8938 is a floor. These arms run to 60.

This asks the obvious follow-up: **does a bigger pretrained backbone pay?** Same recipe
throughout — same MLP head, same encoder lr 1e-5, same cosine warmup, same rotation, same
sampler — so the only thing that changes is the backbone.

| arm | backbone | parameters | size | vs |
|---|---|---|---|---|
| `15_resnet34_finetune` | ResNet34 | 21.4M | 128 | ResNet18: **depth** at the same width |
| `16_resnet50_finetune` | ResNet50 | 24.0M | 128 | ResNet34: **bottlenecks**, 2048-wide features |
| `17_vit_b_32_finetune` | ViT-B/32 | 87.7M | 224 | all of them: **attention, pretrained** |

## Reference points, same splits

| | macro-F1 |
|---|---|
| `v27-resnet_style` from scratch, 2.83M | 0.8900 |
| `v27-convnext_style` from scratch, 414k | 0.8883 |
| `v30-finetune_encoder_1e-5` ResNet18 fine-tuned | **0.8938** (floor; hit the cap) |
| `v29-resnet18_frozen_onehot` frozen | 0.7351 |

## Why ViT-B/32 and not B/16

32x32 patches give 49 tokens instead of 196: phase 0 measured 458 s/epoch against 1482 for
B/16 on a T4. It is the small ViT in the sense that costs time.

**224x224 is not a choice for it.** Torchvision ViT checkpoints carry positional embeddings
fitted to exactly that size and raise on anything else, so this arm cannot share the 128px
geometry of 15 and 16. Two honest consequences: the input is a further upsample of a
212x187 map, and 224² is past the 2 GiB geometry-cache budget so the loader falls back to
redoing letterboxing per access. Both belong in the presentation rather than in a footnote —
the ViT is not being compared on equal terms, and saying so is the correct move.

## What to expect

The from-scratch models top out around 0.89-0.90 with **2.8M** parameters. If a 24M ResNet50
or an 87M ViT lands in the same place, that is the finding: on 121k wafer maps at this
resolution, the ceiling is the data and not the model. Given capacity was already flat
within ConvNeXt (414k -> 2.68M was -0.002), that is the likely outcome — and it is a
stronger closing slide than another tied number.

## 0. Colab web UI only — clone and authenticate

Skip if the repo is already at `/content/fdl-project`.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. Run

Each arm prints the parameter groups the optimizer built, and asserts the input size against
what the backbone requires — a ViT paired with 128px fails here, in a second, rather than on
the first batch after the 2 GB pickle has loaded.

`resnet50` and `vit_b_32` need the backbone support added in this commit
(`src/fdl_project/models/frozen_backbone.py`), so **pull before running**.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.optim import build_optimizer
from fdl_project.training.runner import run_experiment

SERIES = "v32_bigger_pretrained"
RERUN = False
BUDGET_HOURS = None

CONFIG_DIRECTORY = REPO / "configs/train" / SERIES
CONFIGS = sorted(CONFIG_DIRECTORY.glob("*.yaml"))
assert CONFIGS, f"no configs in {CONFIG_DIRECTORY} -- pull the branch"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT / "results.csv"

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},finetune]"]

results = []
if RESULTS_CSV.exists() and not RERUN:
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"resuming: {len(results)} arm(s) done")

done = {r["run"] for r in results}
dataframe = load_wm811k_dataframe(DATASET)
session_started = time.monotonic()

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"
    assert config.model.kwargs["freeze_encoder"] is False, "this series fine-tunes"

    if config.name in done:
        print(f"skip  {config.name}")
        continue
    if BUDGET_HOURS is not None and (time.monotonic() - session_started) / 3600 > BUDGET_HOURS:
        print(f"\nbudget reached -- stopping before {config.name}")
        break

    model = build_model(config.model.name, **config.model.kwargs)
    # A torchvision ViT's positional embeddings are fitted to one input size and
    # it raises on anything else. Catch that here rather than on the first batch
    # after the 2 GB pickle has loaded.
    required = model.required_input_size
    size = tuple(config.data.preprocessing.target_size)
    assert required is None or size == (required, required), (
        f"{config.name}: {config.model.kwargs['architecture']} requires "
        f"{required}x{required}, config says {size}"
    )
    optimizer = build_optimizer(model, config.optimizer)
    trainable = count_trainable_parameters(model)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n=== {config.name}  ({config.model.kwargs['architecture']}, {size[0]}px)")
    for group in optimizer.param_groups:
        count = sum(p.numel() for p in group["params"])
        print(f"    {str(group.get('name')):10} lr={group['lr']:<9g} {count:>12,}")
    encoder_lr = next((g["lr"] for g in optimizer.param_groups if g.get("name") == "encoder"),
                      config.optimizer.kwargs["lr"])
    del model, optimizer

    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
    results.append({
        "run": config.name,
        "backbone": config.model.kwargs["architecture"],
        "px": size[0],
        "total": total,
        "encoder_lr": encoder_lr,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "scratch_f1": round(float(per_class["Scratch"]), 3),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    if HAS_DRIVE:
        shutil.copy2(RESULTS_CSV, DRIVE / f"{SERIES}_results.csv")
    row = results[-1]
    flag = "  <-- still improving at the cap" if row["best_epoch"] >= row["epochs"] - 2 else ""
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"Scratch {row['scratch_f1']:.3f}  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min{flag}")

print(f"\n{len(results)}/{len(CONFIGS)} arms complete -> {RESULTS_CSV}")

## 4. Read it

Set `RESNET18_FINETUNED` from your v30 run first — it is the arm every row here extends, and
without it the comparison is against the wrong thing.

Noise floor **0.02**.

In [ ]:
# Measured on the same splits, same recipe, same MLP head.
REFERENCE = {
    "v30-finetune_encoder_1e-5 (resnet18, 11.3M)": 0.8938,
    "v29-resnet18_frozen_onehot (frozen)":         0.7351,
    "v27-resnet_style (from scratch, 2.83M)":      0.8900,
    "v27-convnext_style (from scratch, 414k)":     0.8883,
}
RESNET18_FINETUNED = 0.8938   # v30-finetune_encoder_1e-5, and a FLOOR:
                              # best at epoch 31 of 40 with patience 10, so
                              # early stopping never fired -- the cap did.
NOISE_FLOOR = 0.02

frame = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
frame["vs_resnet18"] = (frame["macro_f1"] - RESNET18_FINETUNED).round(4)
frame["truncated"] = frame["best_epoch"] >= frame["epochs"] - 2
pd.set_option("display.width", 250)
display(frame)

print(f"Fine-tuned ResNet18 (v30, the arm these extend): {RESNET18_FINETUNED:.4f}")
for label, score in REFERENCE.items():
    if score is not None:
        print(f"  {label:46} {score:.4f}")

print(f"\nDoes a bigger backbone pay? (noise floor {NOISE_FLOOR:.2f})")
for _, row in frame.iterrows():
    verdict = "REAL" if abs(row["vs_resnet18"]) > NOISE_FLOOR else "noise"
    print(f"  {row['run']:26} {row['vs_resnet18']:+.4f} vs ResNet18  "
          f"({row['total']/1e6:.1f}M parameters)  [{verdict}]")

if frame["truncated"].any():
    print("\nStill improving at the cap:", ", ".join(frame.loc[frame["truncated"], "run"]))

## 5. For the presentation

Together with v29 and v30 this gives a complete transfer-learning story on one axis:

1. **frozen ImageNet** — 0.7351, loses badly
2. **fine-tuned ResNet18** — level with the best from-scratch model
3. **bigger fine-tuned backbones** — this notebook

The interesting claim is not which wins. It is that a **414k ConvNeXt trained from scratch**
matches an **87M pretrained transformer**, if that is how it lands. That says the task is
not short of capacity or of features; it is short of data in the rare classes, which is
exactly what the per-class table has said all along — `Near-full` has 30 validation wafers
and `Scratch` is the class every intervention moves.